# 7.2 DPO

DPO（Direct Preference Optimization）绕过奖励模型，直接用偏好数据优化策略模型，简化了对齐流程。

**目的**：简化对齐流程，避免训练奖励模型和PPO的不稳定性

**基本原理**：通过数学推导，将RLHF的优化目标转化为直接在偏好数据上优化的损失函数，无需显式的奖励模型。

**核心公式**：
- L_DPO = -E[log σ(β(log π_θ(y_w|x)/π_ref(y_w|x) - log π_θ(y_l|x)/π_ref(y_l|x)))]
- y_w: 优选回复，y_l: 劣选回复
- β: 温度参数，控制偏离参考策略的程度

**与RLHF的区别**：
- RLHF：训练RM → PPO优化（2步，4个模型）
- DPO：直接在偏好对上优化（1步，2个模型）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimplePolicy(nn.Module):
    def __init__(self, d_input=64, vocab_size=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_input, 128), nn.ReLU(),
            nn.Linear(128, vocab_size)
        )
    def forward(self, x):
       return self.net(x)

policy = SimplePolicy()
ref_policy = SimplePolicy()
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False

x = torch.randn(8, 64)
y_w = torch.randint(0, 50, (8,))
y_l = torch.randint(0, 50, (8,))

logits_policy = policy(x)
logits_ref = ref_policy(x)

log_pi_w = F.log_softmax(logits_policy, dim=-1).gather(1, y_w.unsqueeze(1)).squeeze(1)
log_pi_l = F.log_softmax(logits_policy, dim=-1).gather(1, y_l.unsqueeze(1)).squeeze(1)
log_ref_w = F.log_softmax(logits_ref, dim=-1).gather(1, y_w.unsqueeze(1)).squeeze(1)
log_ref_l = F.log_softmax(logits_ref, dim=-1).gather(1, y_l.unsqueeze(1)).squeeze(1)

beta = 0.1
log_ratio_w = log_pi_w - log_ref_w
log_ratio_l = log_pi_l - log_ref_l
dpo_loss = -F.logsigmoid(beta * (log_ratio_w - log_ratio_l)).mean()

with torch.no_grad():
    implicit_reward_w = beta * log_ratio_w
    implicit_reward_l = beta * log_ratio_l
    accuracy = (implicit_reward_w > implicit_reward_l).float().mean()

print('=== DPO (Direct Preference Optimization) ===')
print(f'Chosen log-ratio: {log_ratio_w[:4].detach().tolist()}')
print(f'Rejected log-ratio: {log_ratio_l[:4].detach().tolist()}')
print(f'DPO loss: {dpo_loss.item():.4f}')
print(f'Preference accuracy: {accuracy:.2%}')
print(f'\nKey: DPO directly optimizes on preference pairs,')
print(f'no reward model or PPO needed. Simpler and more stable.')